# Graph Neural Networks: Visual Message Passing and a Real Comparison

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yfeng-hsm/KI_Geodatenanalyse_SS26/blob/main/lectures/02_deep_learning/notebooks/gnn_visual_message_passing_colab.ipynb)

This tutorial is designed as the GNN part of the deep learning lecture sequence:

1. neural networks and CNNs,
2. transformers,
3. graph neural networks.

The notebook has two goals:

- **Part A:** make one GNN update visible step by step. Students can see node features, edge weights, learnable weights, messages, aggregation, and updated node states.
- **Part B:** use a real graph dataset to compare a feature-only machine learning baseline with a GNN. The point is to make the benefit of neighbourhood information visible.

The code uses PyTorch and PyTorch Geometric so that the conceptual explanation is connected to standard GNN tooling.

## 0. Colab Setup

PyTorch is usually preinstalled in Colab. PyTorch Geometric is installed here because it provides standard graph datasets and layers.

In [ ]:
!pip -q install torch-geometric networkx matplotlib pandas scikit-learn ipywidgets pyvis

In [ ]:
import html as html_module
import math
import random
import tempfile
from pathlib import Path

import ipywidgets as widgets
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from IPython.display import HTML, Math, clear_output, display
from pyvis.network import Network
from torch import nn
from torch_geometric.datasets import Planetoid
from torch_geometric.nn import GCNConv
from torch_geometric.utils import k_hop_subgraph

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE

# Part A: A GNN Update, Made Visible

A basic graph neural network layer updates each node by mixing information from its neighbours.

For a GCN-style layer, one update can be written as:

\[
H^{(1)} = \sigma\left(\hat{A} X W\right)
\]

where:

- \(X\) is the input node feature matrix,
- \(W\) is a learnable weight matrix,
- \(\hat{A}\) is the normalized adjacency matrix with self-loops,
- \(\sigma\) is a nonlinear activation function,
- \(H^{(1)}\) is the updated node representation.

The graph is tiny so every number can be inspected. The next interactive cell is the most important one: click **Update one layer** repeatedly and watch how values and colors change.

In [ ]:
display(Math(r"H^{(k+1)} = \mathrm{ReLU}\left(\hat{A} H^{(k)} W^{(k)}\right)"))
display(Math(r"\hat{A}_{ij} = \frac{1}{\sqrt{d_i d_j}}"))

In [ ]:
toy_edges = [
    (0, 1), (0, 2),
    (1, 2), (1, 3),
    (2, 4),
    (3, 4), (3, 5),
    (4, 5),
]

node_names = {
    0: "A: river",
    1: "B: river",
    2: "C: mixed",
    3: "D: street",
    4: "E: street",
    5: "F: street",
}

X0 = torch.tensor([
    [1.00, 0.10],
    [0.90, 0.20],
    [0.55, 0.55],
    [0.20, 0.90],
    [0.10, 1.00],
    [0.05, 0.95],
], dtype=torch.float32)

W0 = torch.tensor([
    [ 1.20, -0.70],
    [-0.35,  1.10],
], dtype=torch.float32)

G_toy = nx.Graph()
G_toy.add_edges_from(toy_edges)
pos_toy = nx.spring_layout(G_toy, seed=SEED)

In [ ]:
def normalized_adjacency(num_nodes, edges):
    A = torch.zeros((num_nodes, num_nodes), dtype=torch.float32)
    for i, j in edges:
        A[i, j] = 1
        A[j, i] = 1
    A = A + torch.eye(num_nodes)
    degree = A.sum(dim=1)
    d_inv_sqrt = torch.diag(torch.pow(degree, -0.5))
    A_hat = d_inv_sqrt @ A @ d_inv_sqrt
    return A, A_hat, degree


def feature_color(values):
    values = values.detach().cpu().numpy()
    return values[:, 0] - values[:, 1]


def draw_graph_state(features, title, edge_weights=None, cmap="coolwarm"):
    fig, ax = plt.subplots(figsize=(7, 5))
    scalar = feature_color(features)
    nodes = nx.draw_networkx_nodes(
        G_toy,
        pos_toy,
        node_color=scalar,
        cmap=cmap,
        vmin=-1.2,
        vmax=1.2,
        node_size=1150,
        edgecolors="#222",
        linewidths=1.2,
        ax=ax,
    )
    nx.draw_networkx_edges(G_toy, pos_toy, width=1.8, alpha=0.55, ax=ax)
    labels = {i: f"{i}\\n[{features[i,0]:.2f}, {features[i,1]:.2f}]" for i in G_toy.nodes}
    nx.draw_networkx_labels(G_toy, pos_toy, labels=labels, font_size=9, ax=ax)
    if edge_weights:
        nx.draw_networkx_edge_labels(G_toy, pos_toy, edge_labels=edge_weights, font_size=8, ax=ax)
    fig.colorbar(nodes, ax=ax, shrink=0.75, label="feature[0] - feature[1]")
    ax.set_title(title)
    ax.axis("off")
    plt.show()


def feature_table(features, title):
    df = pd.DataFrame(features.detach().cpu().numpy(), columns=["feature_0", "feature_1"])
    df.insert(0, "node", [node_names[i] for i in range(len(df))])
    print(title)
    display(df.round(3))


A_with_self_loops, A_hat, degree = normalized_adjacency(num_nodes=X0.size(0), edges=toy_edges)
feature_table(X0, "Initial node features X")
draw_graph_state(X0, "Initial node features")

## Interactive Node Classification App

This app turns message passing into a small node classification task.

- Each node has three evidence channels: **water**, **road**, and **urban/mixed context**.
- The small lights on the upper-right of each node show those hidden channels; stronger color means stronger evidence.
- The label below each node is the current prediction: `river`, `street`, or `mixed`.
- Red boundary = selected target, yellow boundary = direct neighbours sending messages, black boundary = other nodes.
- Fixed grey edges are the graph structure. They do not update. Animated red dots only show which fixed edges are used for the selected target now.
- After one layer, the target has mixed information from its direct neighbours. After two layers, its neighbours already contain information from their neighbours, so the target indirectly sees a larger receptive field.
- The update keeps 25% of the previous node state, which prevents the wider receptive field from washing out local evidence too quickly.

Use an ambiguous target such as node 4 to see how neighbourhood evidence can make a classification easier.


In [ ]:
CLASS_EDGES = [
    (0, 1), (0, 2), (1, 2),
    (2, 4), (2, 7),
    (3, 4), (3, 5),
    (4, 5), (4, 7),
    (5, 6),
]

CLASS_NODE_NAMES = {
    0: "A: river bank",
    1: "B: river bend",
    2: "C: bridge",
    3: "D: road",
    4: "E: crossing",
    5: "F: street",
    6: "G: street block",
    7: "H: riverside path",
}

TRUE_LABELS = {
    0: "river",
    1: "river",
    2: "mixed",
    3: "street",
    4: "street",
    5: "street",
    6: "street",
    7: "mixed",
}

FEATURE_NAMES = ["water", "road", "urban/mixed"]
FEATURE_COLORS = ["#2563eb", "#ef4444", "#22c55e"]
CLASS_COLORS = {"river": "#2563eb", "street": "#ef4444", "mixed": "#8b5cf6"}

X_CLASS0 = torch.tensor([
    [0.95, 0.05, 0.20],  # clear river
    [0.88, 0.10, 0.20],  # clear river
    [0.55, 0.48, 0.76],  # bridge / mixed
    [0.12, 0.95, 0.55],  # clear street
    [0.48, 0.50, 0.58],  # ambiguous crossing; neighbourhood should help
    [0.10, 0.92, 0.65],  # clear street
    [0.05, 0.96, 0.35],  # clear street
    [0.70, 0.35, 0.82],  # riverside mixed context
], dtype=torch.float32)

CLASS_WEIGHTS = [
    torch.tensor([
        [0.94, 0.03, 0.05],
        [0.04, 1.02, 0.07],
        [0.10, 0.12, 0.84],
    ], dtype=torch.float32),
    torch.tensor([
        [0.90, 0.04, 0.08],
        [0.03, 1.05, 0.08],
        [0.08, 0.14, 0.86],
    ], dtype=torch.float32),
]

# Keep part of the current node state so wider receptive fields do not erase local evidence.
# This is a residual/self-retention variant of the basic GCN update.
RESIDUAL_KEEP = 0.25

NODE_POS = {
    0: (120, 115),
    1: (315, 95),
    2: (235, 245),
    3: (560, 150),
    4: (420, 305),
    5: (610, 340),
    6: (735, 465),
    7: (245, 430),
}

ROLE_STYLE = {
    "target": {"stroke": "#d62728", "width": 7, "label": "selected target"},
    "neighbour": {"stroke": "#f0c419", "width": 6, "label": "direct message source"},
    "inactive": {"stroke": "#111111", "width": 3, "label": "not directly used now"},
}

app_state = {
    "layer": 0,
    "features": X_CLASS0.clone(),
    "history": [X_CLASS0.clone()],
    "last_update": None,
}

target_dropdown = widgets.Dropdown(
    options=[(f"{i}: {CLASS_NODE_NAMES[i]}", i) for i in range(X_CLASS0.size(0))],
    value=4,
    description="Target",
)
update_button = widgets.Button(description="Update one layer", button_style="primary", icon="refresh")
reset_button = widgets.Button(description="Reset", button_style="", icon="undo")
app_output = widgets.Output()


def normalized_row_adjacency(num_nodes, edges):
    A = torch.zeros((num_nodes, num_nodes), dtype=torch.float32)
    for i, j in edges:
        A[i, j] = 1
        A[j, i] = 1
    A = A + torch.eye(num_nodes)
    degree = A.sum(dim=1, keepdim=True)
    return A / degree


A_CLASS = normalized_row_adjacency(X_CLASS0.size(0), CLASS_EDGES)


def direct_neighbours(node):
    return sorted([int(i) for i in range(X_CLASS0.size(0)) if i != node and A_CLASS[node, i] > 0])


def k_hop_nodes(node, hops):
    seen = {node}
    frontier = {node}
    for _ in range(hops):
        nxt = set()
        for n in frontier:
            nxt.update(direct_neighbours(n))
        nxt -= seen
        seen |= nxt
        frontier = nxt
    return seen


def node_role(node, target):
    if node == target:
        return "target"
    if node in direct_neighbours(target):
        return "neighbour"
    return "inactive"


def class_scores(features):
    water = features[:, 0]
    road = features[:, 1]
    urban = features[:, 2]
    balance = 1.0 - torch.clamp(torch.abs(water - road), 0, 1)
    return torch.stack([
        water + 0.12 * urban,
        road + 0.10 * urban,
        0.55 * urban + 0.45 * balance,
    ], dim=1)


def class_probabilities(features):
    return torch.softmax(class_scores(features) * 3.2, dim=1)


def predicted_labels(features):
    names = ["river", "street", "mixed"]
    probs = class_probabilities(features)
    return [names[int(i)] for i in probs.argmax(dim=1)]


def fmt(value):
    return f"{float(value):.2f}"


def clamp01(value):
    return max(0.0, min(1.0, float(value)))


def light_opacity(value):
    return 0.20 + 0.80 * clamp01(value)


def light_radius(value):
    return 5.5 + 6.5 * clamp01(value)


def feature_lights_svg(values, x, y, scale=1.0):
    parts = []
    for idx, value in enumerate(values):
        cx = x + idx * 18 * scale
        radius = light_radius(value) * scale
        opacity = light_opacity(value)
        parts.append(
            f"<circle cx='{cx:.1f}' cy='{y:.1f}' r='{radius:.1f}' fill='{FEATURE_COLORS[idx]}' fill-opacity='{opacity:.2f}' "
            f"stroke='#111827' stroke-width='{1.1 * scale:.1f}'><title>{FEATURE_NAMES[idx]}: {fmt(value)}</title></circle>"
        )
    return "".join(parts)


def mini_lights(values):
    return "".join(
        f"<span class='mini-light' style='background:{FEATURE_COLORS[idx]}; opacity:{light_opacity(value):.2f}' title='{FEATURE_NAMES[idx]}: {fmt(value)}'></span>"
        for idx, value in enumerate(values)
    )


def evidence_bars(values, title):
    rows = []
    for idx, value in enumerate(values):
        width = int(100 * clamp01(value))
        rows.append(f"""
        <div class='bar-row'>
          <div class='bar-label'>{FEATURE_NAMES[idx]}</div>
          <div class='bar-track'><div class='bar-fill' style='width:{width}%; background:{FEATURE_COLORS[idx]};'></div></div>
          <div class='bar-value'>{fmt(value)}</div>
        </div>
        """)
    return f"<div class='evidence-box'><div class='box-title'>{title}</div>{''.join(rows)}</div>"


def prediction_pill(label, prob=None):
    confidence = "" if prob is None else f" {int(prob * 100)}%"
    return f"<span class='pred-pill' style='border-color:{CLASS_COLORS[label]}; background:{CLASS_COLORS[label]}22;'>{label}{confidence}</span>"


def gcn_step(features, weight):
    neighbour_messages = A_CLASS @ (features @ weight)
    return torch.relu(RESIDUAL_KEEP * features + (1.0 - RESIDUAL_KEEP) * neighbour_messages)


def message_rows(features, weight, target):
    transformed = features @ weight
    rows = []
    for src in [target] + direct_neighbours(target):
        coeff = A_CLASS[target, src].item()
        message = coeff * transformed[src]
        rows.append({
            "source": src,
            "source_name": CLASS_NODE_NAMES[src],
            "coeff": coeff,
            "before": features[src].detach().cpu().numpy(),
            "after_w": transformed[src].detach().cpu().numpy(),
            "message": message.detach().cpu().numpy(),
        })
    return rows


def graph_html(features, target, layer):
    preds = predicted_labels(features)
    probs = class_probabilities(features)
    one_hop = set(direct_neighbours(target))
    two_hop_only = k_hop_nodes(target, 2) - k_hop_nodes(target, 1)
    show_two_hop = layer >= 2
    edge_parts = []
    active_parts = []
    node_parts = []
    tx, ty = NODE_POS[target]

    for i, j in CLASS_EDGES:
        x1, y1 = NODE_POS[i]
        x2, y2 = NODE_POS[j]
        edge_parts.append(f"<line x1='{x1}' y1='{y1}' x2='{x2}' y2='{y2}' stroke='#cbd5e1' stroke-width='2.4'/>")
        if target in (i, j):
            src = j if i == target else i
            sx, sy = NODE_POS[src]
            coeff = A_CLASS[target, src].item()
            active_parts.append(f"""
            <line x1='{sx}' y1='{sy}' x2='{tx}' y2='{ty}' stroke='#d62728' stroke-width='4.2' opacity='0.75' marker-end='url(#arrowhead)'/>
            <text x='{(sx + tx) / 2:.1f}' y='{(sy + ty) / 2 - 8:.1f}' class='edge-label'>{coeff:.2f}</text>
            <circle r='7' fill='#d62728'>
              <animateMotion dur='1.35s' begin='{0.12 * len(active_parts):.2f}s' repeatCount='indefinite' path='M {sx},{sy} L {tx},{ty}' />
            </circle>
            """)

    self_coeff = A_CLASS[target, target].item()
    active_parts.append(f"""
    <circle cx='{tx}' cy='{ty}' r='50' fill='none' stroke='#d62728' stroke-width='3' opacity='0.0'>
      <animate attributeName='r' values='40;56;40' dur='1.35s' repeatCount='indefinite'/>
      <animate attributeName='opacity' values='0.15;0.85;0.15' dur='1.35s' repeatCount='indefinite'/>
    </circle>
    <text x='{tx + 42}' y='{ty - 44}' class='edge-label'>self {self_coeff:.2f}</text>
    """)

    if show_two_hop:
        for node in two_hop_only:
            x, y = NODE_POS[node]
            node_parts.append(f"<circle cx='{x}' cy='{y}' r='52' fill='#fef3c7' opacity='0.36' stroke='#f59e0b' stroke-dasharray='6 5' stroke-width='2'/>")

    for node, (x, y) in NODE_POS.items():
        role = node_role(node, target)
        style = ROLE_STYLE[role]
        values = features[node].detach().cpu().numpy()
        pred = preds[node]
        pred_color = CLASS_COLORS[pred]
        prob = probs[node].max().item()
        node_parts.append(f"""
        <g class='node'>
          <circle cx='{x}' cy='{y}' r='42' fill='#ffffff' stroke='{pred_color}' stroke-width='10' opacity='0.25'/>
          <circle cx='{x}' cy='{y}' r='42' fill='#ffffff' stroke='{style['stroke']}' stroke-width='{style['width']}'/>
          <text x='{x}' y='{y + 7}' text-anchor='middle' class='node-id'>{node}</text>
          <text x='{x}' y='{y + 62}' text-anchor='middle' class='node-name'>{html_module.escape(pred)} {int(prob * 100)}%</text>
          <g transform='translate(17,-35)'>{feature_lights_svg(values, x, y, scale=0.82)}</g>
          <title>{node}: {CLASS_NODE_NAMES[node]} | true: {TRUE_LABELS[node]} | predicted: {pred} {int(prob * 100)}% | water {fmt(values[0])}, road {fmt(values[1])}, urban {fmt(values[2])}</title>
        </g>
        """)

    two_hop_note = "2-hop context highlighted" if show_two_hop else "click update again to reveal larger receptive field"
    return f"""
    <style>
      .graph-card {{ border:1px solid #d1d5db; border-radius:8px; background:#fff !important; color:#111827 !important; overflow:hidden; }}
      .node-id {{ font: 900 23px Arial, sans-serif; fill:#111827; }}
      .node-name {{ font: 800 12px Arial, sans-serif; fill:#111827; }}
      .edge-label {{ font: 900 13px Arial, sans-serif; fill:#b91c1c; paint-order:stroke; stroke:#fff; stroke-width:4px; }}
      .legend text {{ font: 800 12px Arial, sans-serif; fill:#111827; }}
    </style>
    <div class='graph-card'>
      <svg viewBox='45 45 760 500' width='100%' height='590' role='img'>
        <defs>
          <marker id='arrowhead' markerWidth='11' markerHeight='9' refX='10' refY='4.5' orient='auto'>
            <path d='M0,0 L11,4.5 L0,9 z' fill='#d62728'></path>
          </marker>
          <filter id='softShadow' x='-20%' y='-20%' width='140%' height='140%'>
            <feDropShadow dx='0' dy='3' stdDeviation='3' flood-color='#94a3b8' flood-opacity='0.28'/>
          </filter>
        </defs>
        <rect x='52' y='52' width='744' height='486' rx='14' fill='#ffffff' stroke='#d1d5db'/>
        <g>{''.join(edge_parts)}</g>
        <g>{''.join(active_parts)}</g>
        <g filter='url(#softShadow)'>{''.join(node_parts)}</g>
        <g transform='translate(70 505)' class='legend'>
          <text x='0' y='0'>fixed graph, animated messages only</text>
          <circle cx='230' cy='-5' r='10' fill='#fff' stroke='#d62728' stroke-width='4'/><text x='246' y='0'>target</text>
          <circle cx='320' cy='-5' r='10' fill='#fff' stroke='#f0c419' stroke-width='4'/><text x='336' y='0'>1-hop</text>
          <circle cx='405' cy='-5' r='10' fill='#fff' stroke='#111' stroke-width='3'/><text x='421' y='0'>other</text>
          <circle cx='510' cy='-5' r='8' fill='{FEATURE_COLORS[0]}'/><text x='523' y='0'>water</text>
          <circle cx='590' cy='-5' r='8' fill='{FEATURE_COLORS[1]}'/><text x='603' y='0'>road</text>
          <circle cx='660' cy='-5' r='8' fill='{FEATURE_COLORS[2]}'/><text x='673' y='0'>urban</text>
          <text x='0' y='24'>{two_hop_note}</text>
        </g>
      </svg>
    </div>
    """


def explanation_html(features, weight, target, layer, next_features):
    probs = class_probabilities(features)
    next_probs = class_probabilities(next_features)
    preds = predicted_labels(features)
    next_preds = predicted_labels(next_features)
    target_values = features[target].detach().cpu().numpy()
    next_values = next_features[target].detach().cpu().numpy()
    rows = message_rows(features, weight, target)
    one_hop = direct_neighbours(target)
    two_hop_only = sorted(k_hop_nodes(target, 2) - k_hop_nodes(target, 1))

    msg_cards = []
    for row in rows:
        border = "#d62728" if row["source"] == target else "#f0c419"
        msg_cards.append(f"""
        <div class='msg-card' style='border-left-color:{border};'>
          <div class='msg-source'>{row['source']}: {html_module.escape(row['source_name'])}</div>
          <div class='msg-line'>{mini_lights(row['before'])}<span>H source</span></div>
          <div class='msg-line'>{mini_lights(row['after_w'])}<span>after W</span></div>
          <div class='msg-line'>{mini_lights(row['message'])}<span>× edge weight {row['coeff']:.2f}</span></div>
        </div>
        """)

    layer_text = {
        0: "Input only: the target is judged mostly from its own three feature lights.",
        1: "Layer 1: the target mixes direct-neighbour messages with 25% of its own previous state. This keeps local evidence visible.",
        2: "Layer 2: neighbours already contain their neighbours' information, so the target indirectly uses 2-hop context without fully over-smoothing.",
    }.get(layer, "More layers keep mixing wider neighbourhood context.")

    pred = preds[target]
    next_pred = next_preds[target]
    pred_prob = probs[target].max().item()
    next_prob = next_probs[target].max().item()

    return f"""
    <style>
      .gnn-card {{ border:1px solid #d1d5db; border-radius:8px; background:#fff !important; color:#111827 !important; padding:12px; height:590px; overflow-y:auto; box-sizing:border-box; }}
      .gnn-card * {{ color:#111827 !important; }}
      .panel-title {{ font-size:18px; font-weight:900; margin-bottom:8px; }}
      .status {{ background:#fff4cc !important; border-left:4px solid #b88700; padding:8px; margin-bottom:10px; font-weight:800; }}
      .formula-strip {{ display:flex; flex-wrap:wrap; align-items:center; gap:6px; margin:9px 0 11px; }}
      .token {{ border:2px solid #111827; border-radius:8px; padding:7px 9px; font-weight:900; background:#fff; }}
      .t-h {{ background:#dbeafe !important; border-color:#2563eb; }}
      .t-w {{ background:#ede9fe !important; border-color:#7c3aed; }}
      .t-a {{ background:#fef3c7 !important; border-color:#f0c419; }}
      .t-relu {{ background:#dcfce7 !important; border-color:#22c55e; }}
      .section-title {{ font-weight:900; margin:10px 0 6px; }}
      .pred-row {{ display:grid; grid-template-columns:1fr 36px 1fr; gap:8px; align-items:center; margin:8px 0; }}
      .pred-box, .evidence-box, .msg-card {{ border:1px solid #d1d5db; border-radius:8px; background:#f9fafb !important; padding:8px; }}
      .pred-box {{ text-align:center; font-weight:900; }}
      .pred-pill {{ display:inline-block; border:3px solid; border-radius:999px; padding:4px 10px; font-weight:900; margin-top:4px; }}
      .arrow {{ text-align:center; font-weight:900; font-size:20px; }}
      .box-title {{ font-weight:900; margin-bottom:5px; }}
      .bar-row {{ display:grid; grid-template-columns:86px 1fr 36px; gap:6px; align-items:center; font-size:12px; margin:4px 0; }}
      .bar-label {{ font-weight:800; }}
      .bar-track {{ height:12px; background:#e5e7eb !important; border-radius:999px; overflow:hidden; }}
      .bar-fill {{ height:100%; border-radius:999px; }}
      .bar-value {{ font-weight:900; text-align:right; }}
      .msg-grid {{ display:grid; grid-template-columns:1fr; gap:7px; }}
      .msg-card {{ border-left:6px solid; }}
      .msg-source {{ font-weight:900; margin-bottom:3px; }}
      .msg-line {{ display:flex; align-items:center; gap:6px; font-size:12px; font-weight:800; margin:2px 0; }}
      .mini-light {{ display:inline-block; width:16px; height:16px; border-radius:50%; border:1px solid #111827; margin-right:2px; }}
      .context-list {{ font-size:12px; line-height:1.45; background:#eef2ff !important; border-radius:8px; padding:8px; font-weight:800; }}
    </style>
    <div class='gnn-card'>
      <div class='panel-title'>Node Classification Logic</div>
      <div class='status'>{layer_text}</div>
      <div class='formula-strip'>
        <span class='token t-h'>25% old H</span><span>+</span>
        <span class='token t-a'>75% neighbour messages</span><span>where</span>
        <span class='token t-h'>H</span><span>→</span>
        <span class='token t-w'>shared W</span><span>→</span>
        <span class='token t-a'>fixed edge weights A</span><span>→</span>
        <span class='token t-relu'>ReLU</span><span>→</span>
        <span class='token'>new H + prediction</span>
      </div>
      <div class='context-list'>
        Target {target}: {html_module.escape(CLASS_NODE_NAMES[target])}<br>
        1-hop neighbours: {', '.join(str(n) for n in one_hop)}<br>
        2-hop nodes that become indirect context after the next layer: {', '.join(str(n) for n in two_hop_only) if two_hop_only else 'none'}<br>
        Edges stay fixed; only hidden evidence and prediction are recalculated.<br>
        This demo keeps 25% of each node's previous state. Without that residual/self-retention, a second layer can over-smooth mixed boundary nodes.
      </div>
      <div class='section-title'>Prediction before and after the next update</div>
      <div class='pred-row'>
        <div class='pred-box'>now<br>{prediction_pill(pred, pred_prob)}{evidence_bars(target_values, 'target evidence')}</div>
        <div class='arrow'>→</div>
        <div class='pred-box'>after update<br>{prediction_pill(next_pred, next_prob)}{evidence_bars(next_values, 'updated evidence')}</div>
      </div>
      <div class='section-title'>Messages used for the selected target</div>
      <div class='msg-grid'>{''.join(msg_cards)}</div>
    </div>
    """


def render_message_passing_app():
    with app_output:
        clear_output(wait=True)
        layer = app_state["layer"]
        features = app_state["features"]
        target = target_dropdown.value
        weight = CLASS_WEIGHTS[min(layer, len(CLASS_WEIGHTS) - 1)]
        next_features = gcn_step(features, weight)
        just_updated = app_state.get("last_update")

        if layer >= len(CLASS_WEIGHTS):
            update_button.description = "Start over"
            update_button.icon = "undo"
        else:
            update_button.description = "Update one layer"
            update_button.icon = "refresh"

        left_panel = widgets.Output(layout=widgets.Layout(width="58%"))
        right_panel = widgets.Output(layout=widgets.Layout(width="42%"))
        display(widgets.HBox([left_panel, right_panel], layout=widgets.Layout(width="100%", align_items="flex-start")))

        with left_panel:
            display(HTML(graph_html(features, target, layer)))

        with right_panel:
            display(HTML(explanation_html(features, weight, target, layer, next_features)))


def update_one_layer(_):
    layer = app_state["layer"]
    if layer >= len(CLASS_WEIGHTS):
        reset_app(None)
        return
    weight = CLASS_WEIGHTS[layer]
    app_state["features"] = gcn_step(app_state["features"], weight)
    app_state["layer"] = layer + 1
    app_state["history"].append(app_state["features"].clone())
    app_state["last_update"] = {"from_layer": layer, "to_layer": layer + 1, "target": target_dropdown.value}
    render_message_passing_app()


def reset_app(_):
    app_state["layer"] = 0
    app_state["features"] = X_CLASS0.clone()
    app_state["history"] = [X_CLASS0.clone()]
    app_state["last_update"] = None
    render_message_passing_app()


def target_changed(_):
    app_state["layer"] = 0
    app_state["features"] = X_CLASS0.clone()
    app_state["history"] = [X_CLASS0.clone()]
    app_state["last_update"] = None
    render_message_passing_app()


update_button.on_click(update_one_layer)
reset_button.on_click(reset_app)
target_dropdown.observe(target_changed, names="value")

controls = widgets.HBox([target_dropdown, update_button, reset_button])
display(widgets.VBox([controls, app_output]))
render_message_passing_app()


## Step 1: Add Self-Loops and Normalize Edges

A GCN keeps a node's own information through a self-loop and normalizes messages by node degree:

\[
\hat{A}_{ij} = \frac{1}{\sqrt{d_i d_j}}
\]

This prevents high-degree nodes from dominating simply because they have many edges.

In [ ]:
display(pd.DataFrame(A_with_self_loops.numpy()).astype(int).style.set_caption("Adjacency with self-loops"))
display(pd.DataFrame(A_hat.numpy()).round(3).style.set_caption("Normalized adjacency A_hat"))

edge_weight_labels = {}
for i, j in G_toy.edges:
    edge_weight_labels[(i, j)] = f"{A_hat[i, j]:.2f}"

draw_graph_state(X0, "Graph with normalized edge weights", edge_weights=edge_weight_labels)

## Step 2: Apply Learnable Weights

Before aggregation, every node feature vector is transformed by the learnable weight matrix `W`. This is the neural-network part of the layer.

In [ ]:
display(pd.DataFrame(W0.numpy(), index=["input_feature_0", "input_feature_1"], columns=["hidden_0", "hidden_1"]).round(3).style.set_caption("Weight matrix W"))

XW = X0 @ W0
feature_table(XW, "Transformed node features X @ W")
draw_graph_state(XW, "After applying learnable weights: X @ W")

## Step 3: Aggregate Neighbour Messages

Now each node receives a weighted sum of transformed features from itself and its neighbours. This is where a GNN differs from a standard MLP.

In [ ]:
H_pre = A_hat @ XW
H1 = torch.relu(H_pre)

feature_table(H_pre, "Before ReLU: A_hat @ X @ W")
feature_table(H1, "After ReLU: H1")

draw_graph_state(H1, "Updated node states after one GCN-style layer")

In [ ]:
def explain_node_update(target_node):
    rows = []
    for source_node in range(X0.size(0)):
        coeff = A_hat[target_node, source_node].item()
        if coeff == 0:
            continue
        transformed = XW[source_node]
        contribution = coeff * transformed
        rows.append({
            "target": node_names[target_node],
            "source": node_names[source_node],
            "normalized_weight": coeff,
            "source_hidden_0": transformed[0].item(),
            "source_hidden_1": transformed[1].item(),
            "message_0": contribution[0].item(),
            "message_1": contribution[1].item(),
        })
    df = pd.DataFrame(rows)
    display(df.round(3))
    print("sum of messages:", H_pre[target_node].detach().numpy().round(3))
    print("after ReLU:", H1[target_node].detach().numpy().round(3))


explain_node_update(target_node=3)

## Step 4: Stack Another Layer

A second GNN layer lets information move two hops. The visualization below shows how representations become smoother across connected neighbourhoods.

In [ ]:
W1 = torch.tensor([
    [0.80, 0.35],
    [0.25, 0.90],
], dtype=torch.float32)

H2_pre = A_hat @ (H1 @ W1)
H2 = torch.relu(H2_pre)

feature_table(H2, "After two GCN-style layers")
draw_graph_state(H2, "Updated node states after two layers")

# Part B: Real Data Comparison - Feature-Only ML vs GNN

The goal is to make a clear teaching comparison:

- **Feature-only MLP:** sees each node's feature vector, but ignores graph edges.
- **GCN:** sees the same node features and additionally uses the graph.

On citation networks such as Cora, neighbouring papers often have related topics. A GNN can exploit this neighbourhood structure.

In [ ]:
dataset = Planetoid(root="/content/pyg_data", name="Cora")
data = dataset[0].to(DEVICE)

print(dataset)
print("nodes:", data.num_nodes)
print("edges:", data.num_edges)
print("node features:", dataset.num_node_features)
print("classes:", dataset.num_classes)
print("train/val/test:", int(data.train_mask.sum()), int(data.val_mask.sum()), int(data.test_mask.sum()))

In [ ]:
class FeatureOnlyMLP(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, dropout=0.5):
        super().__init__()
        self.lin1 = nn.Linear(in_channels, hidden_channels)
        self.lin2 = nn.Linear(hidden_channels, out_channels)
        self.dropout = dropout

    def forward(self, x, edge_index=None):
        x = self.lin1(x)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        return self.lin2(x)


class SimpleGCN(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, dropout=0.5):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, out_channels)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        return self.conv2(x, edge_index)

In [ ]:
@torch.no_grad()
def evaluate(model, data):
    model.eval()
    logits = model(data.x, data.edge_index)
    pred = logits.argmax(dim=-1)
    result = {}
    for split, mask in [
        ("train", data.train_mask),
        ("val", data.val_mask),
        ("test", data.test_mask),
    ]:
        acc = (pred[mask] == data.y[mask]).float().mean().item()
        result[f"{split}_acc"] = acc
    return result


def train_model(model, data, epochs=200, lr=0.01, weight_decay=5e-4):
    model = model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    history = []
    for epoch in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad()
        logits = model(data.x, data.edge_index)
        loss = F.cross_entropy(logits[data.train_mask], data.y[data.train_mask])
        loss.backward()
        optimizer.step()

        metrics = evaluate(model, data)
        metrics["epoch"] = epoch
        metrics["loss"] = loss.item()
        history.append(metrics)
    return model, pd.DataFrame(history)

In [ ]:
torch.manual_seed(SEED)
mlp = FeatureOnlyMLP(dataset.num_node_features, hidden_channels=64, out_channels=dataset.num_classes)
mlp, hist_mlp = train_model(mlp, data, epochs=200)

torch.manual_seed(SEED)
gcn = SimpleGCN(dataset.num_node_features, hidden_channels=64, out_channels=dataset.num_classes)
gcn, hist_gcn = train_model(gcn, data, epochs=200)

summary = pd.DataFrame([
    {"model": "Feature-only MLP", **evaluate(mlp, data)},
    {"model": "GCN with neighbourhood", **evaluate(gcn, data)},
])
display(summary.round(3))

In [ ]:
def plot_training_curves(hist_mlp, hist_gcn):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(hist_mlp["epoch"], hist_mlp["loss"], label="Feature-only MLP")
    axes[0].plot(hist_gcn["epoch"], hist_gcn["loss"], label="GCN")
    axes[0].set_title("Training loss")
    axes[0].set_xlabel("epoch")
    axes[0].legend()

    axes[1].plot(hist_mlp["epoch"], hist_mlp["test_acc"], label="Feature-only MLP")
    axes[1].plot(hist_gcn["epoch"], hist_gcn["test_acc"], label="GCN")
    axes[1].set_title("Test accuracy")
    axes[1].set_xlabel("epoch")
    axes[1].set_ylim(0, 1)
    axes[1].legend()
    plt.show()


plot_training_curves(hist_mlp, hist_gcn)

## Inspect One Prediction and Its Neighbourhood

The next cell selects a test node where the GCN prediction is correct and the feature-only MLP is wrong, if such a node exists. Then it draws the node's 2-hop neighbourhood.

In [ ]:
@torch.no_grad()
def predictions(model):
    model.eval()
    return model(data.x, data.edge_index).argmax(dim=-1).cpu()

mlp_pred = predictions(mlp)
gcn_pred = predictions(gcn)
y_true = data.y.cpu()
test_nodes = data.test_mask.cpu().nonzero(as_tuple=False).view(-1)

candidates = [
    int(node) for node in test_nodes
    if mlp_pred[node] != y_true[node] and gcn_pred[node] == y_true[node]
]

if candidates:
    focus_node = candidates[0]
    print("Selected a test node where GCN is correct and MLP is wrong:", focus_node)
else:
    focus_node = int(test_nodes[0])
    print("No MLP-wrong/GCN-correct node found in this run; showing first test node:", focus_node)

print("true class:", int(y_true[focus_node]))
print("MLP prediction:", int(mlp_pred[focus_node]))
print("GCN prediction:", int(gcn_pred[focus_node]))

In [ ]:
def draw_k_hop_neighbourhood(node_id, num_hops=2):
    subset, edge_index_sub, mapping, edge_mask = k_hop_subgraph(
        node_id,
        num_hops=num_hops,
        edge_index=data.edge_index.cpu(),
        relabel_nodes=True,
    )
    edges = [tuple(edge) for edge in edge_index_sub.t().tolist()]
    G = nx.Graph()
    G.add_edges_from(edges)
    if not G.nodes:
        G.add_node(0)
    pos = nx.spring_layout(G, seed=SEED)

    colors = []
    for local_node in G.nodes:
        original = int(subset[local_node]) if local_node < len(subset) else node_id
        if original == node_id:
            colors.append("#d62728")
        elif int(y_true[original]) == int(y_true[node_id]):
            colors.append("#2ca02c")
        else:
            colors.append("#cccccc")

    labels = {local: str(int(subset[local])) for local in G.nodes if local < len(subset)}
    plt.figure(figsize=(8, 6))
    nx.draw_networkx_edges(G, pos, alpha=0.35)
    nx.draw_networkx_nodes(G, pos, node_color=colors, node_size=260, edgecolors="#222")
    nx.draw_networkx_labels(G, pos, labels=labels, font_size=8)
    plt.title(f"{num_hops}-hop neighbourhood around node {node_id}\\nred = focus node, green = same class")
    plt.axis("off")
    plt.show()


draw_k_hop_neighbourhood(focus_node, num_hops=2)

## Teaching Takeaway

A standard MLP treats every node independently after reading its feature vector. It can learn from features of a paper, but it cannot directly use who cites whom.

A GNN uses the same node features **and** the graph. In citation data, connected papers often share topics. In spatial data, neighbouring cells, nearby street images, adjacent road segments, or spatially connected observations often carry related information. That is why the GNN comparison is a useful bridge to geospatial AI.